# DSPy — Otimização com `InferRules`

Neste notebook será demonstrado o uso do otimizador `InferRules` do DSPy em um problema de **classificação binária de textos**.

Será utilizada a base **Natural Language Processing with Disaster Tweets**, disponibilizada no Kaggle. O objetivo é classificar cada tweet em uma das seguintes categorias:

* `0`: o tweet **não descreve um desastre real**;
* `1`: o tweet **descreve um desastre real**.

O `InferRules` é um otimizador baseado no `BootstrapFewShot` que combina demonstrações few-shot com a **indução de regras explícitas em linguagem natural**.

Em alto nível, o processo utilizado neste notebook é:

```text
trainset
    ↓
BootstrapFewShot
    ↓
demonstrações few-shot
    ↓
indução de regras em linguagem natural
    ↓
múltiplos programas candidatos
    ↓
avaliação interna no valset
    ↓
melhor programa
    ↓
testset
```

O experimento será dividido em três etapas:

1. utilizar o `trainset` para produzir demonstrações e inferir regras;
2. utilizar o `valset` dentro do próprio `InferRules` para selecionar o melhor programa candidato;
3. comparar o baseline e o programa otimizado utilizando apenas o `testset`.

A avaliação final utilizará:

* Accuracy;
* Precision;
* Recall;
* F1-score.

> **Observação:** o F1 será usado como uma das métricas finais de análise. Durante a otimização, será utilizada uma métrica de acerto exato calculável individualmente para cada exemplo.

Referência: https://dspy.ai/api/optimizers/InferRules/

In [1]:
import os
from dotenv import load_dotenv  # Carrega variáveis de ambiente do arquivo .env
import dspy  # Framework para otimização de prompts com Language Models
import pandas as pd

from typing import Literal
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    classification_report,
)

from tqdm.auto import tqdm

/home/leonardo/Documentos/github/dspy_studies/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()  # Lê variáveis do arquivo .env

True

## Setup — Configuração dos modelos

Primeiro, carregamos as variáveis de ambiente, como a API key, do arquivo `.env` na raiz do projeto.

Serão utilizados dois papéis de modelo:

* `lm`: modelo utilizado pelo classificador que será otimizado;
* `teacher_lm`: modelo utilizado durante a otimização, incluindo o bootstrap e a indução das regras.

Neste exemplo, ambos utilizarão `openai/gpt-5-mini`.

O `teacher_lm` será fornecido ao `InferRules` por meio de:

```python
teacher_settings={"lm": teacher_lm}
```

O `InferRules` herda do `BootstrapFewShot` parâmetros como `max_bootstrapped_demos`, `max_labeled_demos`, `max_rounds` e `metric_threshold`.

In [3]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo utilizado para executar o classificador
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Modelo utilizado durante o bootstrap e a indução das regras.
# Para a família GPT-5, utilizamos temperature=1.0.
teacher_lm = dspy.LM(
    "openai/gpt-5-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=1.0,
)

# Define o modelo padrão utilizado pelo programa DSPy
dspy.configure(lm=lm)

## Leitura da base de dados

Será utilizada a base da competição **Natural Language Processing with Disaster Tweets**, do Kaggle.

Referência:

https://www.kaggle.com/competitions/nlp-getting-started

Para este experimento são relevantes as colunas:

* `text`: texto do tweet;
* `target`: classe esperada (`0` ou `1`).

Para reduzir o custo do experimento, serão utilizadas apenas as primeiras **200 linhas** da base.

In [4]:
df = pd.read_csv('disaster_tweets.csv')
df = df.head(200)
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


## Análise da distribuição das classes

Antes da divisão dos dados, é importante verificar quantos exemplos existem de cada classe.

Além da quantidade absoluta, será analisada a proporção entre tweets classificados como `0` e `1`.

Essa análise é importante porque o **F1-score** considera conjuntamente precisão e recall e é especialmente útil quando existe algum grau de desbalanceamento entre as classes.


In [5]:
df["target"].value_counts()

target
0    103
1     97
Name: count, dtype: int64

In [6]:
df["target"].value_counts(normalize=True)

target
0    0.515
1    0.485
Name: proportion, dtype: float64

## Separação entre treino, validação e teste

Os dados serão divididos em três conjuntos independentes:

* **70% para treino**;
* **15% para validação**;
* **15% para teste**.

Cada conjunto possui uma função diferente:

```text
70% trainset
    ↓
bootstrap + indução das regras

15% valset
    ↓
avaliação interna dos candidatos do InferRules

15% testset
    ↓
avaliação final do baseline e do programa otimizado
```

O `trainset` será utilizado pelo `InferRules` para gerar demonstrações e inferir regras.

O `valset` será passado diretamente para `optimizer.compile(...)`. O próprio `InferRules` avalia os candidatos nesse conjunto e retorna o programa com maior score segundo a métrica interna.

O `testset` permanecerá completamente isolado até a avaliação final.

O parâmetro `stratify` preserva aproximadamente a proporção das classes `0` e `1` nos três conjuntos.

> Se `valset` não for informado, a implementação atual do `InferRules` divide o `trainset` recebido em duas partes. Como já criamos um conjunto de validação separado, iremos fornecê-lo explicitamente.

In [7]:
# Primeiro separamos 70% para treino e 30% para validação + teste
df_train, df_temp = train_test_split(
    df[["text", "target"]],
    test_size=0.30,
    random_state=42,
    stratify=df["target"],
)

# Divide os 30% restantes igualmente: 15% validação e 15% teste
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp["target"],
)

print(f"Treino:     {len(df_train)} exemplos")
print(f"Validação:  {len(df_val)} exemplos")
print(f"Teste:      {len(df_test)} exemplos")

Treino:     140 exemplos
Validação:  30 exemplos
Teste:      30 exemplos


## Conversão para `dspy.Example`

O DSPy representa exemplos de treino e teste por meio da classe `dspy.Example`.

Neste problema, cada exemplo possui dois campos:

* `text`: entrada fornecida ao modelo;
* `target`: resposta esperada.

A chamada:

`with_inputs("text")`

informa explicitamente ao DSPy que `text` deve ser utilizado como entrada do programa.

Consequentemente, `target` passa a ser tratado como o **label**, ou seja, a resposta esperada para aquele exemplo.

Conceitualmente, cada registro passa a ter a seguinte estrutura:

`entrada: text → saída esperada: target`

In [8]:
def dataframe_para_dspy(dataframe):
    exemplos = []

    for _, row in dataframe.iterrows():
        exemplo = dspy.Example(
            text=row["text"],
            target=int(row["target"]),
        ).with_inputs("text")

        exemplos.append(exemplo)

    return exemplos

In [9]:
trainset = dataframe_para_dspy(df_train)
valset = dataframe_para_dspy(df_val)
testset = dataframe_para_dspy(df_test)

print(f"Trainset: {len(trainset)}")
print(f"Valset:   {len(valset)}")
print(f"Testset:  {len(testset)}")

Trainset: 140
Valset:   30
Testset:  30


In [10]:
trainset[0]

Example({'text': '13,000 people receive #wildfires evacuation orders in California ', 'target': 1}) (input_keys={'text'})

## Definição da tarefa com uma `Signature`

No DSPy, uma `Signature` descreve declarativamente a tarefa que será executada pelo modelo.

A `ClassificarTweet` possui:

* um `InputField` chamado `text`, contendo o tweet;
* um `OutputField` chamado `target`, contendo a classificação.

O tipo:

`Literal[0, 1]`

restringe a resposta esperada às duas classes válidas do problema.

Dessa forma, a Signature define claramente o contrato:

`texto do tweet → 0 ou 1`


In [11]:
class ClassificarTweet(dspy.Signature):
    """
    Classifique o tweet em uma das duas classes possíveis.
    """

    text: str = dspy.InputField(
        desc="Tweet a ser analisado."
    )

    target: Literal[0, 1] = dspy.OutputField(
        desc="Classe prevista."
    )

In [12]:
classificador_base = dspy.Predict(ClassificarTweet)

In [13]:
def avaliar_classificador(programa, dataset, descricao="Avaliando"):
    """
    Executa um programa DSPy sobre um dataset e calcula
    métricas globais de classificação.
    """

    y_true = []
    y_pred = []

    for exemplo in tqdm(dataset, desc=descricao):

        # Executa o programa utilizando somente os campos
        # marcados como entrada pelo with_inputs(...)
        predicao = programa(**exemplo.inputs())

        # Label verdadeiro
        y_true.append(int(exemplo.target))

        # Label previsto pelo DSPy
        y_pred.append(int(predicao.target))

    # Calcula as métricas sobre todo o conjunto
    resultado = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }

    return resultado

## Baseline no conjunto de validação

Antes da otimização, o classificador original será avaliado no `valset`.

Essa avaliação fornece uma referência do desempenho da instrução original. Posteriormente, o mesmo `valset` será utilizado internamente pelo `InferRules` para selecionar o melhor programa candidato.

O `testset` **ainda não será utilizado**.

In [14]:
resultado_base_validacao = avaliar_classificador(
    classificador_base,
    valset,
    descricao="Baseline no valset",
)

print(f"Accuracy:  {resultado_base_validacao['accuracy']:.4f}")
print(f"Precision: {resultado_base_validacao['precision']:.4f}")
print(f"Recall:    {resultado_base_validacao['recall']:.4f}")
print(f"F1:        {resultado_base_validacao['f1']:.4f}")

Baseline no valset: 100%|█████████| 30/30 [00:01<00:00, 15.69it/s]

Accuracy:  0.8333
Precision: 0.8462
Recall:    0.7857
F1:        0.8148


## Métrica utilizada pelo `InferRules`

O `InferRules` herda seu processo inicial de otimização do `BootstrapFewShot`. A métrica é utilizada para avaliar previsões durante o bootstrap e também para pontuar os programas candidatos no `valset`.

Neste problema será utilizada uma métrica simples de **acerto exato**:

```python
def metrica_infer_rules(example, prediction, trace=None):
    esperado = int(example.target)
    previsto = int(prediction.target)

    return float(esperado == previsto)
```

Ela retorna:

* `1.0` quando a classificação está correta;
* `0.0` quando a classificação está incorreta.

Também será utilizado:

```python
metric_threshold=1.0
```

para que somente previsões totalmente corretas sejam aceitas como demonstrações durante o bootstrap.

Quando essa métrica é agregada sobre o `valset`, o score corresponde à proporção de classificações corretas, isto é, à **accuracy**.

O F1-score não é adequado como métrica independente por exemplo, pois depende conjuntamente dos verdadeiros positivos, falsos positivos e falsos negativos de um conjunto inteiro. Por isso, Accuracy, Precision, Recall e F1 serão calculados separadamente na análise dos resultados.

In [15]:
def metrica_infer_rules(example, prediction, trace=None):
    """
    Métrica utilizada pelo InferRules.

    Retorna 1.0 quando a classe prevista corresponde à classe esperada
    e 0.0 caso contrário.
    """
    esperado = int(example.target)
    previsto = int(prediction.target)

    return float(esperado == previsto)

## Otimização com `InferRules`

O `InferRules` é um otimizador especializado em transformar exemplos em **regras explícitas de decisão**.

Ele estende o `BootstrapFewShot` e combina duas etapas principais:

```text
exemplos de treinamento
        ↓
BootstrapFewShot
        ↓
demonstrações few-shot
        ↓
indução de regras
        ↓
┌─────────────┬─────────────┬─────────────┐
│ candidato 1 │ candidato 2 │ candidato 3 │
│ regras A    │ regras B    │ regras C    │
└─────────────┴─────────────┴─────────────┘
        ↓
avaliação no valset
        ↓
melhor candidato
```

Para cada candidato, o `InferRules` gera regras em linguagem natural e as acrescenta às instruções originais da `Signature`.

Neste experimento serão utilizados:

* `num_candidates=3`: gera três programas candidatos;
* `num_rules=5`: solicita cinco regras para cada preditor;
* `num_threads=4`: permite paralelismo durante a avaliação dos candidatos;
* `max_bootstrapped_demos=4`: limita a quantidade de demonstrações produzidas pelo bootstrap;
* `max_labeled_demos=8`: limita a quantidade de demonstrações rotuladas adicionadas ao programa;
* `max_rounds=1`: limita as tentativas de bootstrap por exemplo;
* `metric_threshold=1.0`: aceita no bootstrap apenas previsões com score máximo.

Foi escolhido um número pequeno de candidatos para reduzir a quantidade de chamadas à API neste experimento didático.

In [16]:
NUM_CANDIDATES = 3
NUM_RULES = 5
MAX_BOOTSTRAPPED_DEMOS = 4
MAX_LABELED_DEMOS = 8
MAX_ROUNDS = 1

optimizer = dspy.InferRules(
    metric=metrica_infer_rules,                 # Métrica usada no bootstrap e na avaliação
    num_candidates=NUM_CANDIDATES,              # Quantidade de programas candidatos
    num_rules=NUM_RULES,                        # Quantidade de regras inferidas por preditor
    num_threads=4,                              # Paralelismo durante a avaliação
    teacher_settings={"lm": teacher_lm},       # LM utilizado durante a otimização
    max_bootstrapped_demos=MAX_BOOTSTRAPPED_DEMOS,
    max_labeled_demos=MAX_LABELED_DEMOS,
    max_rounds=MAX_ROUNDS,
    metric_threshold=1.0,                       # Aceita somente previsões corretas no bootstrap
)

## Compilação do programa

A compilação do `InferRules` recebe:

* `student`: programa DSPy que será otimizado;
* `trainset`: exemplos utilizados no bootstrap e na indução das regras;
* `valset`: exemplos utilizados para avaliar os programas candidatos.

Neste experimento:

```python
classificador_otimizado = optimizer.compile(
    student=classificador_base,
    trainset=trainset,
    valset=valset,
)
```

O fluxo interno pode ser resumido como:

```text
classificador_base
        ↓
BootstrapFewShot
        ↓
indução de regras
        ↓
3 candidatos
        ↓
avaliação no valset
        ↓
melhor candidato
        ↓
classificador_otimizado
```

Diferentemente do notebook com `SIMBA`, não é necessário acessar `candidate_programs` nem fazer uma seleção externa. O próprio `InferRules` avalia os candidatos no `valset` e retorna diretamente o melhor programa.

A assinatura atual de `InferRules.compile()` não possui parâmetro `seed`.

In [17]:
classificador_otimizado = optimizer.compile(
    student=classificador_base,
    trainset=trainset,
    valset=valset,
)

  3%|▊                            | 4/140 [01:03<36:15, 15.99s/it]


Bootstrapped 4 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
Average Metric: 29.00 / 30 (96.7%): 100%|█| 30/30 [00:47<00:00,  1

2026/09/08 10:40:48 INFO dspy.evaluate.evaluate: Average Metric: 29.0 / 30 (96.7%)
2026/09/08 10:40:48 INFO dspy.teleprompt.infer_rules: Evaluated Candidate 1 with score 96.67. Current best score: 96.67



Average Metric: 29.00 / 30 (96.7%): 100%|█| 30/30 [00:46<00:00,  1

2026/09/08 10:42:04 INFO dspy.evaluate.evaluate: Average Metric: 29.0 / 30 (96.7%)
2026/09/08 10:42:04 INFO dspy.teleprompt.infer_rules: Evaluated Candidate 2 with score 96.67. Current best score: 96.67



Average Metric: 29.00 / 30 (96.7%): 100%|█| 30/30 [00:47<00:00,  1

2026/09/08 10:43:14 INFO dspy.evaluate.evaluate: Average Metric: 29.0 / 30 (96.7%)
2026/09/08 10:43:14 INFO dspy.teleprompt.infer_rules: Evaluated Candidate 3 with score 96.67. Current best score: 96.67
2026/09/08 10:43:14 INFO dspy.teleprompt.infer_rules: Final best score: 96.67


## Resultado do programa otimizado no `valset`

Após a compilação, podemos calcular as métricas globais do programa retornado no conjunto de validação.

Esse resultado é útil para inspeção, mas **não deve ser tratado como avaliação final**, pois o próprio `InferRules` utilizou o `valset` para selecionar o melhor candidato.

In [18]:
resultado_otimizado_validacao = avaliar_classificador(
    classificador_otimizado,
    valset,
    descricao="InferRules no valset",
)

comparacao_validacao = pd.DataFrame(
    {
        "Modelo": ["Baseline", "InferRules"],
        "Accuracy": [
            resultado_base_validacao["accuracy"],
            resultado_otimizado_validacao["accuracy"],
        ],
        "Precision": [
            resultado_base_validacao["precision"],
            resultado_otimizado_validacao["precision"],
        ],
        "Recall": [
            resultado_base_validacao["recall"],
            resultado_otimizado_validacao["recall"],
        ],
        "F1": [
            resultado_base_validacao["f1"],
            resultado_otimizado_validacao["f1"],
        ],
    }
)

comparacao_validacao

InferRules no valset: 100%|███████| 30/30 [00:00<00:00, 62.74it/s]


,Modelo,Accuracy,Precision,Recall,F1
0,Baseline,0.833333,0.846154,0.785714,0.814815
1,InferRules,0.966667,1.000000,0.928571,0.962963


## Inspeção das regras inferidas

Uma das características mais interessantes do `InferRules` é que a otimização pode ser inspecionada diretamente.

O otimizador acrescenta as regras descobertas às instruções da `Signature`. Conceitualmente:

```text
instrução original
        ↓
InferRules
        ↓
instrução original + regras inferidas
```

A implementação atual acrescenta às instruções um bloco equivalente a:

```text
Please adhere to the following rules when making your prediction:
[regras inferidas]
```

Como o `InferRules` também herda o processo de `BootstrapFewShot`, o programa otimizado pode conter demonstrações few-shot.

Por isso, serão inspecionados:

1. a instrução original;
2. a instrução após o `InferRules`;
3. as demonstrações armazenadas no preditor.

In [19]:
print("=== INSTRUÇÃO ORIGINAL ===")
print(classificador_base.signature.instructions)

print()
print("=== INSTRUÇÃO APÓS O InferRules ===")
print(classificador_otimizado.signature.instructions)

print()
print("=== DEMONSTRAÇÕES FEW-SHOT ===")
print(f"Quantidade de demos: {len(classificador_otimizado.demos)}")

for i, demo in enumerate(classificador_otimizado.demos, start=1):
    print()
    print(f"Demo {i}:")
    print(demo)

=== INSTRUÇÃO ORIGINAL ===
Classifique o tweet em uma das duas classes possíveis.

Please adhere to the following rules when making your prediction:
1. Label 1 if the tweet reports or describes a real-world emergency, disaster, or accident (traffic crashes, airplane/aircraft or helicopter crashes, fires/wildfires, explosions, shootings, heat-wave deaths, evacuations), whether as a news headline, linked article, eyewitness report, or personal admission of being involved.

2. Label 1 if the tweet conveys incident-related consequences or response: deaths/injuries, people feared killed, evacuation orders, blocked lanes/delays, emergency responders or ambulances on scene, or explicit location/time details about the event.

3. Label 0 if the tweet is conversational, promotional, musical, or uses incident-related words metaphorically or idiomatically (e.g., “ablaze” as a metaphor, “by accident” / “on accident” meaning a mistake), or refers to incidents only figuratively rather than describing

## Avaliação final no `testset`

Somente agora será utilizado o conjunto de teste.

Serão avaliados no **mesmo `testset`**:

* o baseline com a instrução original;
* o programa otimizado pelo `InferRules`.

O `testset` não participou:

* do bootstrap das demonstrações;
* da indução das regras;
* da avaliação dos candidatos;
* da seleção do melhor programa.

Isso permite comparar os dois programas utilizando dados que permaneceram isolados durante toda a otimização.

Serão calculadas:

* Accuracy;
* Precision;
* Recall;
* F1-score.

In [20]:
resultado_base = avaliar_classificador(
    classificador_base,
    testset,
    descricao="Baseline no testset",
)

resultado_otimizado = avaliar_classificador(
    classificador_otimizado,
    testset,
    descricao="InferRules no testset",
)

InferRules no testset: 100%|██████| 30/30 [02:46<00:00,  5.54s/it]


In [21]:
comparacao = pd.DataFrame(
    {
        "Modelo": [
            "Baseline (instrução original)",
            "InferRules",
        ],
        "Accuracy": [
            resultado_base["accuracy"],
            resultado_otimizado["accuracy"],
        ],
        "Precision": [
            resultado_base["precision"],
            resultado_otimizado["precision"],
        ],
        "Recall": [
            resultado_base["recall"],
            resultado_otimizado["recall"],
        ],
        "F1": [
            resultado_base["f1"],
            resultado_otimizado["f1"],
        ],
    }
)

comparacao

,Modelo,Accuracy,Precision,Recall,F1
0,Baseline (instrução original),0.933333,0.882353,1.0,0.937500
1,InferRules,0.966667,0.937500,1.0,0.967742


In [22]:
print("BASELINE")
print(
    classification_report(
        resultado_base["y_true"],
        resultado_base["y_pred"],
        digits=4,
    )
)

print("InferRules")
print(
    classification_report(
        resultado_otimizado["y_true"],
        resultado_otimizado["y_pred"],
        digits=4,
    )
)

BASELINE
              precision    recall  f1-score   support

           0     1.0000    0.8667    0.9286        15
           1     0.8824    1.0000    0.9375        15

    accuracy                         0.9333        30
   macro avg     0.9412    0.9333    0.9330        30
weighted avg     0.9412    0.9333    0.9330        30

InferRules
              precision    recall  f1-score   support

           0     1.0000    0.9333    0.9655        15
           1     0.9375    1.0000    0.9677        15

    accuracy                         0.9667        30
   macro avg     0.9688    0.9667    0.9666        30
weighted avg     0.9688    0.9667    0.9666        30



## Salvando o programa otimizado

Neste ponto, `classificador_otimizado` representa o melhor programa selecionado **internamente pelo `InferRules` no `valset`**.

Como a arquitetura é simples e baseada em `dspy.Predict`, podemos salvar seu estado em JSON:

```python
classificador_otimizado.save("InferRules.json")
```

Esse arquivo preserva o estado do programa, incluindo instruções modificadas e demonstrações few-shot.

Para carregar posteriormente, recriamos a mesma arquitetura Python e aplicamos `.load()`.

In [23]:
classificador_otimizado.save("InferRules.json")

In [24]:
# Recria a mesma arquitetura do programa
classificador_carregado = dspy.Predict(ClassificarTweet)

# Carrega o estado otimizado pelo InferRules
classificador_carregado.load("InferRules.json")

classificador_carregado

Predict(StringSignature(text -> target
    instructions='Classifique o tweet em uma das duas classes possíveis.\n\nPlease adhere to the following rules when making your prediction:\n1. Label 1 if the tweet reports or describes a real-world emergency, disaster, or accident (traffic crashes, airplane/aircraft or helicopter crashes, fires/wildfires, explosions, shootings, heat-wave deaths, evacuations), whether as a news headline, linked article, eyewitness report, or personal admission of being involved.\n\n2. Label 1 if the tweet conveys incident-related consequences or response: deaths/injuries, people feared killed, evacuation orders, blocked lanes/delays, emergency responders or ambulances on scene, or explicit location/time details about the event.\n\n3. Label 0 if the tweet is conversational, promotional, musical, or uses incident-related words metaphorically or idiomatically (e.g., “ablaze” as a metaphor, “by accident” / “on accident” meaning a mistake), or refers to incidents onl

## Teste com um novo tweet

Por fim, podemos executar o programa carregado sobre um novo exemplo e inspecionar a última chamada realizada ao modelo.

In [25]:
tweet = "My phone battery died right before the meeting, what a disaster!"

predicao = classificador_carregado(
    text=tweet
)

print(predicao)

Prediction(
    target=0
)


In [26]:
# Mostra a última chamada ao modelo (n=1 significa uma chamada)
dspy.inspect_history(n=1)





[2026-09-08T10:49:14.604469]

System message:

Your input fields are:
1. `text` (str): Tweet a ser analisado.
Your output fields are:
1. `target` (Literal[0, 1]): Classe prevista.
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## text ## ]]
{text}

Outputs will be a JSON object with the following fields.

{
  "target": "{target}        # note: the value you produce must exactly match (no extra characters) one of: 0; 1"
}
In adhering to this structure, your objective is: 
        Classifique o tweet em uma das duas classes possíveis.
        
        Please adhere to the following rules when making your prediction:
        1. Label 1 if the tweet reports or describes a real-world emergency, disaster, or accident (traffic crashes, airplane/aircraft or helicopter crashes, fires/wildfires, explosions, shootings, heat-wave deaths, evacuations), whether as a news headline, linked article, eye